# 第九阶段第二课：小型 Transformer 实现

把注意力 + 位置编码 + 前馈网络 + 残差连接拼起来，就是一个 Transformer。这一课实现一个能生成文本的字符级小型 Transformer。

## 1. 位置编码

注意力本身不关心词的位置，所以要给每个位置加上位置信息。用 sin/cos 编码。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def position_encoding(seq_len, d_model):
    pe = torch.zeros(seq_len, d_model)
    pos = torch.arange(seq_len).unsqueeze(1).float()
    div = torch.exp(torch.arange(0, d_model, 2).float() * (-torch.log(torch.tensor(10000.0)) / d_model))
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe

pe = position_encoding(10, 32)
print(pe.shape)     # 10 个位置，每个 32 维

## 2. Transformer 块：注意力 + 前馈 + 残差 + 归一化

一个块 = 多头注意力 + LayerNorm + 前馈网络，中间都用残差连接。先补上多头注意力的定义（第一课学过，这里独立可运行）。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=16, n_head=4):
        super().__init__()
        self.n_head = n_head
        self.d_head = d_model // n_head
        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)
        self.Wo = nn.Linear(d_model, d_model)

    def forward(self, x):
        batch, seq, _ = x.shape
        q = self.Wq(x).view(batch, seq, self.n_head, self.d_head).transpose(1, 2)
        k = self.Wk(x).view(batch, seq, self.n_head, self.d_head).transpose(1, 2)
        v = self.Wv(x).view(batch, seq, self.n_head, self.d_head).transpose(1, 2)
        scores = q @ k.transpose(-1, -2) / (self.d_head ** 0.5)
        weights = F.softmax(scores, dim=-1)
        attn = weights @ v
        attn = attn.transpose(1, 2).contiguous().view(batch, seq, -1)
        return self.Wo(attn)

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_head, d_ff):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_head)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))   # 残差 + 注意力
        x = x + self.ffn(self.norm2(x))    # 残差 + 前馈
        return x

block = TransformerBlock(d_model=32, n_head=4, d_ff=64)
x = torch.randn(2, 8, 32)
print(block(x).shape)

## 3. 组装完整的 Transformer 语言模型

输入一串字符，预测下一个字符。用 one-hot 或嵌入把字符变成向量，经过几个 Transformer 块，输出每个位置下一个字符的概率。

In [ ]:
class CharTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=32, n_head=4, n_layers=2, d_ff=64):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos = position_encoding(64, d_model)
        self.blocks = nn.Sequential(*[TransformerBlock(d_model, n_head, d_ff) for _ in range(n_layers)])
        self.out = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        # x: (batch, seq_len) 的字符编号
        x = self.embed(x) + self.pos[:x.size(1)].unsqueeze(0)
        x = self.blocks(x)
        return self.out(x)   # (batch, seq_len, vocab_size)

## 4. 准备字符数据并训练

用一小段文本，让模型学会预测下一个字符。

In [ ]:
text = "hello world! this is a tiny transformer learning to generate text. "
chars = sorted(set(text))
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}
vocab_size = len(chars)
print(f"字符表大小：{vocab_size}，字符：{chars}")

In [ ]:
def make_sequences(text, seq_len=16):
    ids = [char_to_idx[c] for c in text]
    xs, ys = [], []
    for i in range(len(ids) - seq_len):
        xs.append(ids[i:i+seq_len])
        ys.append(ids[i+1:i+seq_len+1])   # 每个位置的"下一个字符"
    return torch.tensor(xs), torch.tensor(ys)

seq_len = 16
xs, ys = make_sequences(text, seq_len)
print("样本数：", xs.shape, "标签形状：", ys.shape)

In [ ]:
model = CharTransformer(vocab_size, d_model=32, n_head=4, n_layers=2, d_ff=64)
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
criterion = nn.CrossEntropyLoss()

for step in range(300):
    pred = model(xs)                       # (N, 16, vocab)
    loss = criterion(pred.reshape(-1, vocab_size), ys.reshape(-1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 100 == 0:
        print(f"step {step}, loss = {loss.item():.3f}")

## 5. 用模型生成文本

从第一个字符开始，每步预测下一个字符，把预测结果接回去继续预测。

In [ ]:
def generate(model, start, length=50):
    model.eval()
    ids = [char_to_idx[c] for c in start]
    with torch.no_grad():
        for _ in range(length):
            input_ids = torch.tensor(ids[-seq_len:]).unsqueeze(0)
            logits = model(input_ids)[0, -1]
            next_id = logits.argmax().item()
            ids.append(next_id)
    return "".join(idx_to_char[i] for i in ids)

print(generate(model, "hello", 50))

## 6. 练习（自己动手写）

练习 1：把训练步数从 300 改到 800，再生成一次文本，看是否更像原文。

练习 2：换一段更长的文本训练（自己写一段 200 字符的中文或英文），看模型能否学到一些规律。

In [ ]:
# 在这里写你的练习代码
